### RAG Pipelines- Data Ingestion to Vector DB Pipeline


In [1]:

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\tusha\AppData\Local\Temp\ipykernel_8540\160271900.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: pdf-sample_0.pdf
  ✓ Loaded 1 pages

Processing: Tushar Kumar Resume.pdf
  ✓ Loaded 2 pages

Total documents loaded: 3


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'OpenOffice.org 2.1', 'creator': 'Writer', 'creationdate': '2007-02-23T17:56:37+02:00', 'author': 'Evangelos Vlachogiannis', 'source': '..\\data\\pdf\\pdf-sample_0.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'pdf-sample_0.pdf', 'file_type': 'pdf'}, page_content='Dummy PDF file\nMachine learning is a subset of artificial intelligence that enables \nsystems to learn and improve\nfrom experience without being explicitly programmed. It focuses \non developing computer programs\nthat can access data and use it to learn for themselves.\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and \npenalties'),
 Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Tushar Kumar HCL Resume', 'source': '..\\data\\pdf\\Tushar Kumar Resume.p

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
## text splitting get into chunks

# # chunk_size = 500


# all_chunks = []

# for document in all_documents:
#     text = document.page_content
#     start = 0

#     while start < len(text):
#         end = min(start + chunk_size, len(text))

#         chunk = text[start:end]

#         all_chunks.append({
#             "text": chunk,
#             "metadata": document.metadata
#         })

#         if end == len(text):
#             break

#         start = end - chunk_overlap

# print("Total chunks:", len(all_chunks))
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance."""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    # example of chunks 
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs



In [6]:
chunks=split_documents(all_pdf_documents)
chunks 

Split 3 documents into 8 chunks

Example chunk:
Content: Dummy PDF file
Machine learning is a subset of artificial intelligence that enables 
systems to learn and improve
from experience without being explicitly programmed. It focuses 
on developing compute...
Metadata: {'producer': 'OpenOffice.org 2.1', 'creator': 'Writer', 'creationdate': '2007-02-23T17:56:37+02:00', 'author': 'Evangelos Vlachogiannis', 'source': '..\\data\\pdf\\pdf-sample_0.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'pdf-sample_0.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'OpenOffice.org 2.1', 'creator': 'Writer', 'creationdate': '2007-02-23T17:56:37+02:00', 'author': 'Evangelos Vlachogiannis', 'source': '..\\data\\pdf\\pdf-sample_0.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'pdf-sample_0.pdf', 'file_type': 'pdf'}, page_content='Dummy PDF file\nMachine learning is a subset of artificial intelligence that enables \nsystems to learn and improve\nfrom experience without being explicitly programmed. It focuses \non developing computer programs\nthat can access data and use it to learn for themselves.\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and \npenalties'),
 Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Tushar Kumar HCL Resume', 'source': '..\\data\\pdf\\Tushar Kumar Resume.p

### embedding and vectorDB


In [ ]:
###  for embeeding we use setnece_tranformer and vector store faiss-cpu or chromedb
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\tusha\Desktop\RAG_practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    # constructor 
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): # all-MiniLM-L6-v2 responsible for converting txt to vector
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 721.60it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\tusha\AppData\Local\Temp\ipykernel_26548\148936987.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0
